# Kalman Strategy Ledger v1 — Colab

Neon `market_price` + `strategy_signal` → next-bar execution → fee/slippage → BACKTEST ledger → equity curve → performance metrics.

**Default:** `WRITE_BACK=False` / no Toss trading.


In [ ]:
!pip -q install "psycopg[binary]>=3.2" pandas numpy matplotlib sqlalchemy


In [ ]:
import os, uuid, warnings
from dataclasses import dataclass
from typing import Optional, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text
warnings.filterwarnings('ignore')

MARKET='US'           # US / KR / CRYPTO
SYMBOLS=None          # e.g. ['QQQ','SPY']
START_DATE='2017-01-01'
END_DATE=None
INITIAL_CAPITAL=1_000_000.0
POSITION_FRACTION=0.10
MAX_OPEN_POSITIONS=5
COMMISSION_BPS=5.0
SLIPPAGE_BPS=5.0
MAX_HOLD_DAYS=20
WRITE_BACK=False
BUY_LABELS={'BUY','LONG','ENTER','1'}
SELL_LABELS={'SELL','EXIT','CLOSE','-1'}
print('CONFIG LOADED')


In [ ]:
DATABASE_URL=os.getenv('NEON_DATABASE_URL')
if not DATABASE_URL:
    from google.colab import userdata
    DATABASE_URL=userdata.get('NEON_DATABASE_URL')
assert DATABASE_URL, 'Colab Secrets에 NEON_DATABASE_URL을 추가하세요.'
engine=create_engine(DATABASE_URL,pool_pre_ping=True,future=True)
with engine.connect() as conn:
    print('Neon connected:',conn.execute(text('select now()')).scalar())


In [ ]:
def table_columns(table_name,schema='public'):
    q=text("SELECT column_name FROM information_schema.columns WHERE table_schema=:schema AND table_name=:table ORDER BY ordinal_position")
    return pd.read_sql(q,engine,params={'schema':schema,'table':table_name})['column_name'].tolist()

def pick_col(cols,cands,required=True):
    m={c.lower():c for c in cols}
    for x in cands:
        if x.lower() in m:return m[x.lower()]
    if required: raise KeyError(f'Missing {cands}; available={cols}')
    return None

pc=table_columns('market_price')
sc=table_columns('strategy_signal')
P={
 'symbol':pick_col(pc,['symbol','ticker']),
 'ts':pick_col(pc,['ts','timestamp','datetime','date','as_of','bar_time']),
 'open':pick_col(pc,['open','open_price']),
 'high':pick_col(pc,['high','high_price'],False),
 'low':pick_col(pc,['low','low_price'],False),
 'close':pick_col(pc,['close','close_price','price']),
 'market':pick_col(pc,['market'],False),
 'timeframe':pick_col(pc,['timeframe','interval','frequency'],False)}
S={
 'symbol':pick_col(sc,['symbol','ticker']),
 'ts':pick_col(sc,['as_of','ts','timestamp','created_at','signal_time','date']),
 'signal':pick_col(sc,['signal','side','action','decision']),
 'market':pick_col(sc,['market'],False),
 'entry_allowed':pick_col(sc,['entry_allowed'],False),
 'model_id':pick_col(sc,['model_id'],False),
 'run_id':pick_col(sc,['run_id'],False)}
print('PRICE_SCHEMA',P); print('SIGNAL_SCHEMA',S)


In [ ]:
def _where(schema,start,end,market,symbols):
    w=[f"{schema['ts']} >= :start"]; p={'start':start}
    if end: w.append(f"{schema['ts']} <= :end"); p['end']=end
    if market and schema.get('market'): w.append(f"{schema['market']}=:market"); p['market']=market
    if symbols:
        qs=[]
        for i,sym in enumerate(symbols): qs.append(f':s{i}'); p[f's{i}']=sym
        w.append(f"{schema['symbol']} IN ({','.join(qs)})")
    return w,p

w,p=_where(P,START_DATE,END_DATE,MARKET,SYMBOLS)
pcols=list(dict.fromkeys([x for x in [P['symbol'],P['ts'],P['open'],P['high'],P['low'],P['close'],P['market'],P['timeframe']] if x]))
prices_raw=pd.read_sql(text(f"SELECT {','.join(pcols)} FROM market_price WHERE {' AND '.join(w)} ORDER BY {P['symbol']},{P['ts']}"),engine,params=p)

w,p=_where(S,START_DATE,END_DATE,MARKET,SYMBOLS)
scols=list(dict.fromkeys([x for x in S.values() if x]))
signals_raw=pd.read_sql(text(f"SELECT {','.join(scols)} FROM strategy_signal WHERE {' AND '.join(w)} ORDER BY {S['symbol']},{S['ts']}"),engine,params=p)

prices=pd.DataFrame({'symbol':prices_raw[P['symbol']].astype(str),'ts':pd.to_datetime(prices_raw[P['ts']],utc=True),'open':pd.to_numeric(prices_raw[P['open']],errors='coerce'),'close':pd.to_numeric(prices_raw[P['close']],errors='coerce')})
prices['high']=pd.to_numeric(prices_raw[P['high']],errors='coerce') if P['high'] else prices[['open','close']].max(axis=1)
prices['low']=pd.to_numeric(prices_raw[P['low']],errors='coerce') if P['low'] else prices[['open','close']].min(axis=1)
if P['timeframe']:
    tf=prices_raw[P['timeframe']].astype(str).str.lower()
    m=tf.isin(['1d','d','day','daily','1day','24h'])
    if m.any(): prices=prices[m].copy()
prices=prices.dropna().drop_duplicates(['symbol','ts'],keep='last').sort_values(['symbol','ts']).reset_index(drop=True)

signals=pd.DataFrame({'symbol':signals_raw[S['symbol']].astype(str),'signal_ts':pd.to_datetime(signals_raw[S['ts']],utc=True),'signal':signals_raw[S['signal']].astype(str).str.upper().str.strip()})
signals['entry_allowed']=signals_raw[S['entry_allowed']].fillna(False).astype(bool) if S['entry_allowed'] else True
signals['model_id']=signals_raw[S['model_id']] if S['model_id'] else None
signals['run_id']=signals_raw[S['run_id']] if S['run_id'] else None
signals=signals.dropna(subset=['symbol','signal_ts','signal']).sort_values(['signal_ts','symbol']).reset_index(drop=True)
print('prices',prices.shape,'signals',signals.shape)
display(signals.tail(20))


In [ ]:
def attach_next_bar(signals,prices):
    rows=[]; pmap={s:g.sort_values('ts').reset_index(drop=True) for s,g in prices.groupby('symbol')}
    for r in signals.itertuples(index=False):
        if r.symbol not in pmap: continue
        p=pmap[r.symbol]; i=p['ts'].searchsorted(r.signal_ts,side='right')
        if i>=len(p): continue
        b=p.iloc[i]; d=r._asdict(); d.update({'fill_ts':b.ts,'fill_open':float(b.open),'fill_close':float(b.close)})
        rows.append(d)
    return pd.DataFrame(rows)
mapped=attach_next_bar(signals,prices)
print('mapped',mapped.shape)
display(mapped.head(20))


In [ ]:
@dataclass
class Position:
    symbol:str; qty:float; entry_ts:pd.Timestamp; entry_price:float; entry_fee:float; model_id:Optional[str]=None; signal_run_id:Optional[str]=None

def buy_fill(px): return float(px)*(1+SLIPPAGE_BPS/10000)
def sell_fill(px): return float(px)*(1-SLIPPAGE_BPS/10000)
def fee(n): return abs(float(n))*COMMISSION_BPS/10000

def simulate(mapped,prices):
    cash=float(INITIAL_CAPITAL); pos={}; trades=[]; eq=[]
    pmap={s:g.set_index('ts').sort_index() for s,g in prices.groupby('symbol')}
    dates=sorted(pd.to_datetime(prices.ts,utc=True).unique())
    smap={pd.Timestamp(t):g.copy() for t,g in mapped.groupby('fill_ts')}
    for ts in dates:
        ts=pd.Timestamp(ts); g=smap.get(ts)
        if g is not None:
            for r in g.itertuples(index=False):
                if str(r.signal).upper() in SELL_LABELS and r.symbol in pos:
                    p=pos.pop(r.symbol); xp=sell_fill(r.fill_open); ef=fee(p.qty*xp); cash+=p.qty*xp-ef
                    gp=p.qty*(xp-p.entry_price); npnl=gp-p.entry_fee-ef
                    trades.append({'trade_id':str(uuid.uuid4()),'mode':'BACKTEST','market':MARKET,'symbol':r.symbol,'entry_ts':p.entry_ts,'exit_ts':ts,'qty':p.qty,'entry_price':p.entry_price,'exit_price':xp,'entry_fee':p.entry_fee,'exit_fee':ef,'gross_pnl':gp,'net_pnl':npnl,'return_pct':xp/p.entry_price-1,'exit_reason':'SELL_SIGNAL','model_id':p.model_id,'signal_run_id':p.signal_run_id})
            for r in g.itertuples(index=False):
                if str(r.signal).upper() not in BUY_LABELS or not bool(r.entry_allowed) or r.symbol in pos or len(pos)>=MAX_OPEN_POSITIONS: continue
                mv=0.0
                for s,p in pos.items():
                    pg=pmap.get(s); px=float(pg.loc[ts,'close']) if pg is not None and ts in pg.index else p.entry_price; mv+=p.qty*px
                budget=min(cash,(cash+mv)*POSITION_FRACTION)
                if budget<=0: continue
                ep=buy_fill(r.fill_open); q=budget/ep; ef=fee(q*ep); total=q*ep+ef
                if total>cash:
                    q=max(0.0,(cash/(1+COMMISSION_BPS/10000))/ep); ef=fee(q*ep); total=q*ep+ef
                if q<=0: continue
                cash-=total; pos[r.symbol]=Position(r.symbol,q,ts,ep,ef,getattr(r,'model_id',None),getattr(r,'run_id',None))
        # max holding exit
        forced=[]
        for s,p in list(pos.items()):
            if MAX_HOLD_DAYS is not None and (ts.normalize()-p.entry_ts.normalize()).days>=MAX_HOLD_DAYS and s in pmap and ts in pmap[s].index:
                forced.append((s,float(pmap[s].loc[ts,'close'])))
        for s,raw in forced:
            p=pos.pop(s); xp=sell_fill(raw); ef=fee(p.qty*xp); cash+=p.qty*xp-ef; gp=p.qty*(xp-p.entry_price); npnl=gp-p.entry_fee-ef
            trades.append({'trade_id':str(uuid.uuid4()),'mode':'BACKTEST','market':MARKET,'symbol':s,'entry_ts':p.entry_ts,'exit_ts':ts,'qty':p.qty,'entry_price':p.entry_price,'exit_price':xp,'entry_fee':p.entry_fee,'exit_fee':ef,'gross_pnl':gp,'net_pnl':npnl,'return_pct':xp/p.entry_price-1,'exit_reason':'MAX_HOLD','model_id':p.model_id,'signal_run_id':p.signal_run_id})
        pv=0.0
        for s,p in pos.items():
            pg=pmap.get(s); px=float(pg.loc[ts,'close']) if pg is not None and ts in pg.index else p.entry_price; pv+=p.qty*px
        eq.append({'ts':ts,'cash':cash,'position_value':pv,'equity':cash+pv,'open_positions':len(pos)})
    return pd.DataFrame(trades),pd.DataFrame(eq)

trades,equity=simulate(mapped,prices)
print('trades',len(trades)); display(trades.head(20)); display(equity.tail())


In [ ]:
def metrics(trades,equity):
    if equity.empty:return {}
    e=equity.sort_values('ts').copy(); e['ret']=e.equity.pct_change().fillna(0)
    s=float(e.equity.iloc[0]); z=float(e.equity.iloc[-1]); total=z/s-1
    years=max(1,(e.ts.iloc[-1]-e.ts.iloc[0]).days)/365.25; cagr=(z/s)**(1/years)-1
    sd=e.ret.std(ddof=0); sharpe=e.ret.mean()/sd*np.sqrt(252) if sd>0 else np.nan
    d=e.ret[e.ret<0].std(ddof=0); sortino=e.ret.mean()/d*np.sqrt(252) if pd.notna(d) and d>0 else np.nan
    dd=e.equity/e.equity.cummax()-1; mdd=float(dd.min()); calmar=cagr/abs(mdd) if mdd<0 else np.nan
    pnl=pd.to_numeric(trades.net_pnl,errors='coerce').fillna(0) if not trades.empty else pd.Series(dtype=float)
    gp=float(pnl[pnl>0].sum()) if len(pnl) else 0; gl=abs(float(pnl[pnl<0].sum())) if len(pnl) else 0
    return {'market':MARKET,'start':e.ts.iloc[0],'end':e.ts.iloc[-1],'initial_capital':INITIAL_CAPITAL,'ending_equity':z,'total_return':total,'cagr':cagr,'annualized_volatility':float(sd*np.sqrt(252)),'sharpe':float(sharpe) if pd.notna(sharpe) else np.nan,'sortino':float(sortino) if pd.notna(sortino) else np.nan,'max_drawdown':mdd,'calmar':float(calmar) if pd.notna(calmar) else np.nan,'win_rate':float((pnl>0).mean()) if len(pnl) else np.nan,'profit_factor':gp/gl if gl>0 else np.nan,'trade_count':len(trades),'exposure':float((e.position_value>0).mean()),'commission_bps':COMMISSION_BPS,'slippage_bps':SLIPPAGE_BPS}

m=pd.DataFrame([metrics(trades,equity)]); display(m.T)
if not equity.empty:
    fig,ax=plt.subplots(figsize=(12,4)); ax.plot(equity.ts,equity.equity); ax.set_title(f'Kalman Backtest Equity — {MARKET}'); ax.grid(True,alpha=.25); plt.show()
    dd=equity.set_index('ts').equity/equity.set_index('ts').equity.cummax()-1
    fig,ax=plt.subplots(figsize=(12,3)); ax.plot(dd.index,dd.values); ax.set_title('Drawdown'); ax.grid(True,alpha=.25); plt.show()


In [ ]:
BACKTEST_ID=str(uuid.uuid4()); OUT='/content/kalman_strategy_ledger_v1'; os.makedirs(OUT,exist_ok=True)
trades['backtest_id']=BACKTEST_ID; equity['backtest_id']=BACKTEST_ID; m['backtest_id']=BACKTEST_ID
trades.to_csv(f'{OUT}/strategy_ledger_backtest.csv',index=False)
equity.to_csv(f'{OUT}/strategy_daily_equity.csv',index=False)
m.to_csv(f'{OUT}/strategy_performance.csv',index=False)
print('BACKTEST_ID',BACKTEST_ID)
print('WRITE_BACK',WRITE_BACK)
if WRITE_BACK:
    trades.to_sql('strategy_ledger_backtest_v1',engine,if_exists='append',index=False)
    equity.to_sql('strategy_daily_equity_v1',engine,if_exists='append',index=False)
    m.to_sql('strategy_performance_v1',engine,if_exists='append',index=False)
    print('Neon write-back complete')
